In [ ]:
# Testing for hw1 for Berkeley
from __future__ import annotations

import sys
from pathlib import Path
from datetime import datetime

import abc
from typing import Literal, TypeAlias
import torch
import numpy as np
from torch.utils.data import DataLoader
from dataclasses import asdict, dataclass

import tyro
import wandb

_hw1_src = Path("..") / "berkeley" / "homework_spring2026" / "hw1" / "src"
sys.path.insert(0, str(_hw1_src.resolve()))


from hw1_imitation.data import Normalizer, PushtChunkDataset, download_pusht, load_pusht_zarr
from hw1_imitation.model import BasePolicy
from hw1_imitation.evaluation import Logger, evaluate_policy


In [ ]:
PolicyType: TypeAlias = Literal["mse", "flow"]
LOGDIR_PREFIX = "exp"

@dataclass
class TrainConfig:
    # The path to download the Push-T dataset to.
    data_dir: Path = Path("data")

    # The policy type -- either MSE or flow.
    policy_type: PolicyType = "mse"
    # The number of denoising steps to use for the flow policy (has no effect for the MSE policy).
    flow_num_steps: int = 10
    # The action chunk size.
    chunk_size: int = 8

    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 0.0
    hidden_dims: tuple[int, ...] = (256, 256, 256)
    # The number of epochs to train for.
    num_epochs: int = 400
    # How often to run evaluation, measured in training steps.
    eval_interval: int = 10_000
    num_video_episodes: int = 5
    video_size: tuple[int, int] = (256, 256)
    # How often to log training metrics, measured in training steps.
    log_interval: int = 100
    # Random seed.
    seed: int = 42
    # WandB project name.
    wandb_project: str = "hw1-imitation"
    # Experiment name suffix for logging and WandB.
    exp_name: str | None = None


def parse_train_config(
    args: list[str] | None = None,
    *,
    defaults: TrainConfig | None = None,
    description: str = "Train a Push-T MLP policy.",
) -> TrainConfig:
    defaults = defaults or TrainConfig()
    return tyro.cli(
        TrainConfig,
        args=args,
        default=defaults,
        description=description,
    )


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def config_to_dict(config: TrainConfig) -> dict[str, Any]:
    data = asdict(config)
    for key, value in data.items():
        if isinstance(value, Path):
            data[key] = str(value)
    return data

In [ ]:
config = parse_train_config(args=[])
config_dict = config_to_dict(TrainConfig())

In [ ]:
from __future__ import annotations

import abc
from typing import Literal, TypeAlias


from torch import nn
import torch.optim as optim
import torch
import einops




class MSEPolicy(BasePolicy):
    """Predicts action chunks with an MSE loss."""

    ### TODO: IMPLEMENT MSEPolicy HERE ###
    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        chunk_size: int,
        hidden_dims: tuple[int, ...] = (128, 128),
    ) -> None:
        super().__init__(state_dim, action_dim, chunk_size)
        self.net = MSENet(state_dim, chunk_size, action_dim, hidden_dims)

    def compute_loss(self, state: torch.Tensor, action_chunk: torch.Tensor,) -> torch.Tensor:
        y = self.sample_actions(state)
        total_loss = (y - action_chunk) ** 2
        summed_loss = total_loss.sum(dim=(1, 2))
        average_loss = summed_loss.mean()
        return average_loss
        #raise NotImplementedError

    def sample_actions(
        self,
        state: torch.Tensor,
        *,
        num_steps: int = 10,
    ) -> torch.Tensor:
        return self.net.forward(state)
        # raise NotImplementedError

class MSENet(nn.Module):
    def __init__(self, input_dim, chunk_size, action_dim, hidden_dims):
        super().__init__()
        self.input_dim = input_dim
        self.chunk_size = chunk_size
        self.action_dim = action_dim
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, self.chunk_size*self.action_dim))

        self.net = nn.Sequential(*layers)


    def forward(self, x):
        y = self.net(x)
        y = einops.rearrange(y, 'b (t a) -> b t a', t=self.chunk_size, a=self.action_dim)
        return(y)


In [ ]:
def run_training(config: TrainConfig) -> None:
    set_seed(config.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    zarr_path = download_pusht(config.data_dir)
    states, actions, episode_ends = load_pusht_zarr(zarr_path)
    normalizer = Normalizer.from_data(states, actions)

    dataset = PushtChunkDataset(
        states,
        actions,
        episode_ends,
        chunk_size=config.chunk_size,
        normalizer=normalizer,
    )

    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=True,
    )
    model = MSEPolicy(states.shape[1], actions.shape[1], config.chunk_size, hidden_dims=config.hidden_dims).to(device)

    # model = build_policy(
    #     config.policy_type,
    #     state_dim=states.shape[1],
    #     action_dim=actions.shape[1],
    #     chunk_size=config.chunk_size,
    #     hidden_dims=config.hidden_dims,
    # ).to(device)

    exp_name = f"seed_{config.seed}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    if config.exp_name is not None:
        exp_name += f"_{config.exp_name}"
    log_dir = Path(LOGDIR_PREFIX) / exp_name
    wandb.init(
        project=config.wandb_project, config=config_to_dict(config), name=exp_name
    )
    logger = Logger(log_dir)

    ### TODO: PUT YOUR MAIN TRAINING LOOP HERE ###
    optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay,)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, drop_last=True,)
    for epoch in range(config.num_epochs):  # loop over the dataset multiple times
        running_loss = 0.0
        print(f"Epoch: {epoch}")
        for i, data in enumerate(loader, 0):
        # get the inputs; data is a list of [inputs, labels]
            obs, actions = data
            obs = obs.to(device)
            actions = actions.to(device)
            #print(i)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            #outputs = model.sample_actions(obs)
            #print(f'outputs.shape: {outputs.shape}, actions: {actions.shape}')
            loss = model.compute_loss(obs, actions)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            if i % 2000 == 19:    # print every 2000 mini-batches
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
                running_loss = 0.0

    print('Finished Training')
    #logger.dump_for_grading()
    evaluate_policy(model=model, device=device, chunk_size=config.chunk_size, normalizer=normalizer, video_size=config.video_size, num_video_episodes=config.num_video_episodes, flow_num_steps=config.flow_num_steps, logger=logger, step=0)
    print("finished Evaluation")
    logger.dump_for_grading()


In [ ]:
run_training(config)

In [ ]:
def criterion(y: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    total_loss = (y - actions) ** 2
    summed_loss = total_loss.sum(dim=(1, 2))
    average_loss = summed_loss.mean()
    return average_loss

In [ ]:
model = MSEPolicy(5, 2, 8)
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2,
)

for epoch in range(2):  # loop over the dataset multiple times
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
    )
    running_loss = 0.0
    for i, data in enumerate(loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        obs, actions = data
        print(i)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        #outputs = model.sample_actions(obs)
        #print(f'outputs.shape: {outputs.shape}, actions: {actions.shape}')
        loss = model.compute_loss(obs, actions)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 20 == 19:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 20:.3f}')
            running_loss = 0.0

print('Finished Training')

In [ ]:
model.forward(obs).shape

In [ ]:
# Loss computation:
y = model.sample_actions(obs)
print(f'y.shape = {y.shape}, actions.shape = {actions.shape}')


total_loss = (y - actions)**2
summed_loss = total_loss.sum(dim=(1,2))
average_loss = summed_loss.mean()
print(f'total_loss: {total_loss.shape}, summed_loss: {summed_loss.shape}, average_loss: {average_loss}')

In [ ]:
# First attempt at training MSEModel

for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

In [ ]:
from torch import nn
import torch
import gymnasium
import pygame
import numpy as np

In [ ]:
x = torch.zeros(3, 4)
y = torch.randn(10, device='cpu')

a = torch.tensor([1, 2, 3])
b = torch.as_tensor(a)

w = torch.ones(5, dtype=torch.float64)

torch.manual_seed(0)


In [ ]:
w = torch.randn(3, requires_grad=True)
x = torch.randn(3)

y = (w*x).sum()
y.backward()

print( w.grad, x.grad)

In [ ]:

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return(logits)


In [ ]:
device="cpu"
model = NeuralNetwork().to(device)
print(model)

In [ ]:
X = torch.randn(10, 1, 28, 28, device=device)
logits = model(X)

In [ ]:
logits.size()
logits


In [ ]:
pred_probs = nn.Softmax(dim=1)(logits)
pred_probs.argmax(dim=1)

In [ ]:
print(torch.backends.mps.is_available())